In [1]:
import customtkinter as ctk
import psycopg2
from tkinter import ttk
import csv
from tkinter import filedialog

ctk.set_appearance_mode("light")

# DATABASE CONNECTION
def connect():
    return psycopg2.connect(
        dbname="safepath",
        user="postgres",
        password="cos101",
        host="localhost",
        port="5432"
    )

    

def export_csv():
    conn = connect()
    cur = conn.cursor()

    cur.execute("""
        SELECT id, username, location, incident_type, urgency
        FROM reports
        ORDER BY id ASC
    """)

    rows = cur.fetchall()

    file_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV Files", "*.csv")]
    )

    if file_path:
        with open(file_path, mode="w", newline="") as file:
            writer = csv.writer(file)

            writer.writerow([
                "ID",
                "Username",
                "Location",
                "Incident Type",
                "Urgency"
            ])

            writer.writerows(rows)

    cur.close()
    conn.close()

# FETCH RECORDS
def load_reports(search_text=""):
    conn = connect()
    cur = conn.cursor()

    query = """
        SELECT id, username, location, incident_type, urgency
        FROM reports
        WHERE
            username ILIKE %s OR
            location ILIKE %s OR
            incident_type ILIKE %s
        ORDER BY id ASC
    """

    value = f"%{search_text}%"

    cur.execute(query, (value, value, value))

    rows = cur.fetchall()

    for row in tree.get_children():
        tree.delete(row)

    for row in rows:
        tree.insert("", "end", values=row)

    cur.close()
    conn.close()
# MAIN WINDOW
root = ctk.CTk()
root.geometry("1200x700")
root.title("App Records")

# COLORS
sidebar_color = "#4B1E0F"
main_bg = "#F3EBDD"
button_color = "#C47A3A"

# SIDEBAR
sidebar = ctk.CTkFrame(
    root,
    width=220,
    fg_color=sidebar_color,
    corner_radius=0
)

sidebar.pack(side="left", fill="y")

logo = ctk.CTkLabel(
    sidebar,
    text="SafePath",
    font=("Georgia", 28, "bold"),
    text_color="white"
)

logo.pack(pady=40)

menu_items = [
    "Dashboard",
    "Situation Check",
    "Report Case",
    "Records",
    "Insights",
    "Resources",
    "Echoes of Home",
    "Settings"
]

for item in menu_items:
    btn = ctk.CTkButton(
        sidebar,
        text=item,
        fg_color=button_color if item == "Records" else "transparent",
        hover_color="#8A4F27",
        anchor="w"
    )
    btn.pack(fill="x", padx=20, pady=8)

# MAIN CONTENT
main_frame = ctk.CTkFrame(
    root,
    fg_color=main_bg,
    corner_radius=0
)

main_frame.pack(side="right", fill="both", expand=True)

title = ctk.CTkLabel(
    main_frame,
    text="Records",
    font=("Georgia", 32, "bold"),
    text_color="#3E2A1F"
)

title.pack(anchor="nw", padx=40, pady=(30,10))

# SEARCH FRAME
search_frame = ctk.CTkFrame(
    main_frame,
    fg_color="transparent"
)

search_frame.pack(fill="x", padx=40, pady=(10,0))

# SEARCH ENTRY
search_entry = ctk.CTkEntry(
    search_frame,
    width=300,
    placeholder_text="Search records..."
)

search_entry.pack(side="left", padx=(0,10))

# SEARCH BUTTON
search_button = ctk.CTkButton(
    search_frame,
    text="Search",
    fg_color="#5A2D0C",
    hover_color="#7A431A",
    text_color="white",
    command=lambda: load_reports(search_entry.get())
)

search_button.pack(side="left", padx=10)
# EXPORT BUTTON
export_button = ctk.CTkButton(
    search_frame,
    text="Export as CSV",
    fg_color="#5A2D0C",
    hover_color="#7A431A",
    text_color="white",
    command=export_csv
)

export_button.pack(side="right")

# TABLE FRAME
table_frame = ctk.CTkFrame(
    main_frame,
    fg_color="#F7F0E4"
)

table_frame.pack(fill="both", expand=True, padx=40, pady=30)


# TABLE
columns = ("ID", "Username", "Location", "Incident Type", "Urgency")
style = ttk.Style()

style.theme_use("default")

style.configure(
    "Treeview",
    background="#F3E7D7",
    foreground="#3E2A1F",
    fieldbackground="#F3E7D7",
    rowheight=35,
    bordercolor="#6B3E26",
    borderwidth=1,
    font=("Arial", 11)
)

style.configure(
    "Treeview.Heading",
    background="#5A2D0C",
    foreground="white",
    font=("Georgia", 12, "bold")
)

style.map(
    "Treeview",
    background=[("selected", "#C47A3A")]
)

tree = ttk.Treeview(
    table_frame,
    columns=columns,
    show="headings",
    height=20
)

for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=150)

tree.pack(fill="both", expand=True, padx=20, pady=20)

# LOAD DATA
load_reports()

root.mainloop()